In [9]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
import pandas as pd
from langchain_chroma import Chroma
import torch
# print(torch.__version__, torch.version.cuda)
# print(torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [10]:
#importing dataset

df = pd.read_csv("../datasets/pubmed abstracts/pubmed_abstracts copy.csv")
df.head()

,Unnamed: 0,deep_learning
0,0,(['Magnetic resonance spectroscopic imaging (M...
1,1,(['Existing deep convolutional neural networks...
2,2,(['Deep learning techniques have been increasi...
3,3,(['The original article unfortunately containe...
4,4,(['The most common applications of artificial ...


In [11]:
#loading dataset in langchain

docs = []
for col in df.columns:
    if col == "Unnamed: 0":
        continue
    text = "\n".join(df[col].dropna().astype(str))
    docs.append(Document(page_content=text, metadata={"column": col}))

print(len(docs))   # = number of columns
print(docs[0].metadata)
print(type(docs[0].page_content))

1
{'column': 'deep_learning'}
<class 'str'>


In [12]:
#Spliting text

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(docs)
print(len(texts)) 
print(texts[5].page_content)

22842
by the classification branch. Feeding a chest radiograph to the trained Thorax-Net, a diagnosis is obtained by averaging and binarizing the outputs of two branches. The proposed Thorax-Net model has been evaluated against three state-of-the-art deep learning models using the patientwise official split of the ChestX-ray14 dataset and against other five deep learning models using the imagewise random data split. Our results show that Thorax-Net achieves an average per-class area under the receiver operating characteristic curve (AUC) of 0.7876 and 0.896 in both experiments, respectively, which are higher than the AUC values obtained by other deep models when they were all trained with no external data.'], 'Thorax-Net: An Attention Regularized Deep Neural Network for Classification of Thoracic Diseases on Chest Radiography.')


In [13]:
#Embeddings Model 

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"batch_size": 512},
    )

embeddings._client.half()                 # fp16
embeddings._client.max_seq_length = 256   # truncate long abstracts

query_embedding = embeddings.embed_query("What is a sentence embedding?")
doc_embeddings = embeddings.embed_documents(
    [
        "Sentence embeddings map text to dense vectors.",
        "LangChain provides a standard Embeddings interface.",
    ]
)

print(type(embeddings))
print(hasattr(embeddings, "_client"), hasattr(embeddings, "client"))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 529.69it/s]


<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
True False


In [7]:
m = embeddings._client
print(m.max_seq_length)                  # 256
print(next(m.parameters()).dtype)        # torch.float16
print(next(m.parameters()).device)       # cuda:0

256
torch.float16
cuda:0


In [4]:
#Embeddigns & Vector storage

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

vector_store.add_documents(texts)

NameError: name 'texts' is not defined

In [ ]:
#Timeline of storage

from tqdm import tqdm

max_bs = vector_store._client.get_max_batch_size()   # 5461 for you
BATCH = min(2000, max_bs)                            # stay safely under it

for i in tqdm(range(0, len(texts), BATCH)):
    batch = texts[i:i + BATCH]
    ids = [str(d.metadata.get("pmid", i + j)) for j, d in enumerate(batch)]
    vector_store.add_documents(batch, ids=ids)

100%|██████████| 12/12 [1:02:40<00:00, 313.33s/it]


In [14]:
# Retrieve evidence for the same question that will be evaluated.
query = "What is ADHD?"
retrieved_docs = retriever.invoke(query)
retrieval_context = [doc.page_content for doc in retrieved_docs]
if not retrieval_context:
    raise ValueError("The retriever returned no context; cannot evaluate a RAG answer.")
print(f"Retrieved {len(retrieval_context)} context chunks.")


Retrieved 4 context chunks.


In [23]:
#Setting up LLM

llm = ChatOllama(
    model="qwen3.5:2b", 
    temperature = 0.5,
    reasoning=False,
    
)

In [24]:
# Generate the RAG answer from the retrieved evidence.
context_text = "\n\n".join(retrieval_context)
prompt = f"""Answer the question using only the supplied context. If the context does not contain the answer, say so.

Context:
{context_text}

Question: {query}
Answer:"""
response = llm.invoke(prompt)
print(response.content)


ADHD (Attention Deficit Hyperactivity Disorder) is a neurodevelopmental disorder characterized by inattention, hyperactivity, and impulsive behaviors. It affects young children, who often have difficulty keeping still and exhibit increased fine and gross motor activity. Diagnosis is mainly dependent on behavioral analysis rather than complete understanding of its underlying mechanisms.


In [50]:
import ollama
r = ollama.chat(model="gemma4:e4b", messages=[{"role": "user", "content": "Say hi in JSON: {\"msg\": ...}"}])
print(repr(r.message.content))

'```json\n{"msg": "Hi there!"}\n```'


In [25]:
import os

# Gemma runs mostly on CPU on this machine, so allow longer local judge calls.
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "600"

from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric,
)
from deepeval.models import OllamaModel
from deepeval.test_case import LLMTestCase

# Use a separate DeepEval judge; do not overwrite the LangChain generation model.
evaluation_model = OllamaModel(
    model="qwen3.5:2b",
    base_url="http://localhost:11434",
    temperature=0,
)

test_case = LLMTestCase(
    input=query,
    actual_output=response.content,
    context=retrieval_context,
    retrieval_context=retrieval_context,
)

metrics = {
    "hallucination": HallucinationMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "faithfulness": FaithfulnessMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
    "answer relevancy": AnswerRelevancyMetric(
        threshold=0.5, model=evaluation_model, include_reason=True, async_mode=False
    ),
}

for name, metric in metrics.items():
    metric.measure(test_case)
    print(f"{name}: {metric.score:.3f}")
    print(f"  passed: {metric.is_successful()}")
    print(f"  reason: {metric.reason}")


s:\Projects\rag-dissertation\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

ValidationError: 1 validation error for Verdicts
  Invalid JSON: EOF while parsing a value at line 1 column 0 [type=json_invalid, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/json_invalid